<a href="https://colab.research.google.com/github/lipsasahu-oss/python/blob/main/Resume_Screening_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI-Based Resume Screening System

This Google Colab project provides a simple AI-based system to screen resumes. It extracts text from a PDF resume, applies basic NLP techniques, and evaluates candidates based on required skills, qualifications, experience, and academic percentage.

### Project Goal:
The system will determine if a candidate is 'Selected / Shortlisted' or 'Rejected' based on predefined criteria.

## 1. Setup and Library Installations
First, we need to install the necessary Python libraries: `PyPDF2` for PDF text extraction, `nltk` for natural language processing, and `pandas` (though not explicitly used for core logic, it's a common data handling library in such projects and included in requirements).


In [ ]:
# Install necessary libraries
!pip install PyPDF2
!pip install nltk
!pip install pandas

print("Libraries installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.7 MB/s eta 0:00:00
Libraries installed successfully!


## 2. Import Libraries and Download NLTK Data
Next, we import the installed libraries and download essential NLTK data like `punkt` (for tokenization) and `stopwords` (for removing common words).

In [ ]:
import PyPDF2
import nltk
import re
import pandas as pd
from io import BytesIO
from google.colab import files

# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab') # Added to resolve LookupError
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

print("Libraries imported and NLTK data downloaded!")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Libraries imported and NLTK data downloaded!


## 3. Helper Functions
We define helper functions to extract text from PDF files and to clean/preprocess the extracted text for analysis.

In [ ]:
def extract_text_from_pdf(pdf_file):
    """
    Extracts text from a given PDF file.
    Args:
        pdf_file: A file-like object (BytesIO) containing the PDF data.
    Returns:
        str: Extracted text from the PDF.
    """
    text = ""
    try:
        reader = PyPDF2.PdfReader(pdf_file)
        for page_num in range(len(reader.pages)):
            page = reader.pages[page_num]
            text += page.extract_text() if page.extract_text() else ""
    except Exception as e:
        print(f"Error extracting text from PDF: {e}")
        return ""
    return text

def clean_text(text):
    """
    Cleans the input text by converting to lowercase, removing punctuation,
    tokenizing, and removing stopwords.
    Args:
        text (str): The input text.
    Returns:
        str: Cleaned text.
    """
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(filtered_tokens)

print("Helper functions defined!")

Helper functions defined!


## 4. Resume Analysis Logic
This section defines the core logic for analyzing the resume, including:
- **Required Skills:** Checking for specific programming languages and domains.
- **Valid Qualifications:** Identifying acceptable academic degrees.
- **Experience Levels:** Recognizing different career stages.
- **Academic Percentage:** Extracting and validating the percentage score.
- **Match Percentage Calculation:** Quantifying the candidate's fit.
- **Final Decision:** Determining 'Selected' or 'Rejected' based on criteria.

In [ ]:
# Define screening criteria
REQUIRED_SKILLS = ['python', 'java', 'machine learning', 'sql']
VALID_QUALIFICATIONS = ['graduate', 'btech', 'bsc', 'mca']
VALID_EXPERIENCE_LEVELS = ['intern', 'fresher', 'experience']
MIN_ACADEMIC_PERCENTAGE = 60

def check_skills(text_tokens):
    """
    Checks for required skills in the tokenized resume text.
    Args:
        text_tokens (list): List of tokens from the resume text.
    Returns:
        tuple: (list of found skills, boolean indicating if at least one skill is found)
    """
    found_skills = []
    text_str = ' '.join(text_tokens)
    for skill in REQUIRED_SKILLS:
        if skill in text_str:
            found_skills.append(skill)
    return found_skills, len(found_skills) > 0

def check_qualification(text_tokens):
    """
    Checks for valid qualifications in the tokenized resume text.
    Args:
        text_tokens (list): List of tokens from the resume text.
    Returns:
        str: The found qualification or 'N/A' if none found.
    """
    text_str = ' '.join(text_tokens)
    for qual in VALID_QUALIFICATIONS:
        if qual in text_str:
            return qual.replace('btech', 'B.Tech').replace('bsc', 'BSc').replace('mca', 'MCA').title()
    return "N/A"

def check_experience(text_tokens):
    """
    Checks for experience level in the tokenized resume text.
    Args:
        text_tokens (list): List of tokens from the resume text.
    Returns:
        str: The found experience level or 'N/A' if none found.
    """
    text_str = ' '.join(text_tokens)
    for exp in VALID_EXPERIENCE_LEVELS:
        if exp in text_str:
            return exp.title()
    return "N/A"

def extract_percentage(text):
    """
    Extracts academic percentage from the resume text using regex.
    Args:
        text (str): The raw resume text.
    Returns:
        int: The extracted percentage as an integer, or 0 if not found.
    """
    # Regex to find percentages (e.g., 75%, 75.5%) or scores like 8.5/10 which is 85%
    # Prioritize 'X%' format, then 'X.X/10', then 'X/10'
    percentage_matches = re.findall(r'\b(\d{1,3}(?:\.\d{1,2})?)\s*%', text)
    if percentage_matches:
        # Take the first plausible percentage, assume it's academic
        return int(float(percentage_matches[0]))

    # Try to find 'X.X/10' or 'X/10' and convert to percentage
    gpa_matches = re.findall(r'\b(\d(?:\.\d)?)/10\b', text)
    if gpa_matches:
        return int(float(gpa_matches[0]) * 10)

    return 0

def calculate_match_percentage(found_skills, has_qualification, academic_percentage, has_experience):
    """
    Calculates a match percentage based on skills, qualification, and academic percentage.
    Args:
        found_skills (list): List of skills found.
        has_qualification (bool): True if a valid qualification is found.
        academic_percentage (int): Extracted academic percentage.
        has_experience (bool): True if an experience level is found (used as a factor).
    Returns:
        int: The calculated match percentage.
    """
    score = 0
    max_score = 100

    # Skills contribute significantly
    if len(found_skills) > 0:
        score += (len(found_skills) / len(REQUIRED_SKILLS)) * 40 # 40% for skills

    # Qualification contributes
    if has_qualification:
        score += 20 # 20% for qualification

    # Academic Percentage contributes
    if academic_percentage >= MIN_ACADEMIC_PERCENTAGE:
        score += 30 # 30% for meeting percentage criteria

    # Experience contributes
    if has_experience:
        score += 10 # 10% for having an experience level mentioned

    return min(int(score), max_score)

def make_final_decision(has_min_percentage, has_qualification, has_required_skill):
    """
    Determines the final hiring decision.
    Args:
        has_min_percentage (bool): True if academic percentage meets minimum.
        has_qualification (bool): True if a valid qualification is found.
        has_required_skill (bool): True if at least one required skill is found.
    Returns:
        str: "Selected / Shortlisted" or "Rejected".
    """
    if has_min_percentage and has_qualification and has_required_skill:
        return "Selected / Shortlisted"
    else:
        return "Rejected"

print("Resume analysis logic functions defined!")

Resume analysis logic functions defined!


## 5. Sample Resume Text (for quick testing)
Here's a sample resume text you can use if you don't have a PDF readily available. Copy-paste this into a PDF and upload, or modify the main execution block to use this string directly for testing.

In [ ]:
SAMPLE_RESUME_TEXT = """
John Doe
Software Engineer

Contact:
Email: john.doe@email.com
Phone: (123) 456-7890
LinkedIn: linkedin.com/in/johndoe

Summary:
A highly motivated and skilled B.Tech Graduate with 3 years of experience in software development, specializing in Python and Java. Eager to apply machine learning techniques to solve complex problems. Achieved 85% in academics.

Experience:
Senior Software Engineer | Tech Solutions Inc. | 2021 - Present
- Developed and maintained scalable web applications using Python and Django.
- Implemented RESTful APIs and integrated with various databases including SQL.
- Led a small team of developers on several projects.

Software Intern | Innovate Corp. | 2020 - 2021
- Assisted in the development of machine learning models.
- Wrote comprehensive unit tests and documentation.

Education:
B.Tech in Computer Science | University of Technology | 2017 - 2021
Academic Percentage: 85%

Skills:
Programming Languages: Python, Java, JavaScript, C++
Frameworks: Django, Flask, Spring Boot
Databases: SQL, MongoDB
Tools: Git, Docker, Kubernetes
Machine Learning: TensorFlow, Keras, Scikit-learn
"""

print("Sample resume text loaded.")

Sample resume text loaded.


## 6. Main Execution Block
This is the main part of the system. It handles:
1.  **File Upload:** Allows you to upload a PDF resume from your local machine.
2.  **Text Extraction:** Extracts text from the uploaded PDF.
3.  **Text Cleaning:** Preprocesses the extracted text.
4.  **Analysis:** Applies all the defined logic to check for skills, qualifications, experience, and academic percentage.
5.  **Decision & Output:** Calculates match percentage and makes a final decision, then displays the results in a clear, professional format.

In [ ]:
def run_screening_system(resume_pdf_content=None, use_sample=False):
    """
    Runs the resume screening system.
    Args:
        resume_pdf_content (bytes): Content of the PDF file if uploaded via Colab.
        use_sample (bool): If True, uses SAMPLE_RESUME_TEXT instead of uploaded PDF.
    """

    raw_resume_text = ""
    if use_sample:
        raw_resume_text = SAMPLE_RESUME_TEXT
        print("Using sample resume text for screening.")
    elif resume_pdf_content:
        print("Extracting text from uploaded PDF...")
        raw_resume_text = extract_text_from_pdf(BytesIO(resume_pdf_content))
        if not raw_resume_text:
            print("Could not extract text from PDF. Please check the file.")
            return
    else:
        print("No resume provided. Please upload a PDF or set `use_sample=True`.")
        return

    # NLP Processing
    cleaned_text = clean_text(raw_resume_text)
    text_tokens = word_tokenize(cleaned_text)
    # For debugging: print(f"Cleaned Text Tokens: {text_tokens}")

    # 1. Skills Check
    found_skills, has_required_skill = check_skills(text_tokens)

    # 2. Qualification Check
    qualification = check_qualification(text_tokens)
    has_qualification = qualification != "N/A"

    # 3. Experience Check
    experience = check_experience(text_tokens)
    has_experience = experience != "N/A"

    # 4. Academic Percentage
    academic_percentage = extract_percentage(raw_resume_text) # Use raw text for percentage regex
    has_min_percentage = academic_percentage >= MIN_ACADEMIC_PERCENTAGE

    # 5. Match Percentage Logic
    match_percentage = calculate_match_percentage(found_skills, has_qualification, academic_percentage, has_experience)

    # 6. Final Decision Logic
    final_decision = make_final_decision(has_min_percentage, has_qualification, has_required_skill)

    # Output Results
    print("\n" + "="*35)
    print("RESUME SCREENING RESULT".center(35))
    print("="*35)

    print(f"Extracted Skills: {', '.join(skill.title() for skill in found_skills) if found_skills else 'None'}")
    print(f"Qualification: {qualification}")
    print(f"Experience: {experience}")
    print(f"Academic Percentage: {academic_percentage}% (Min required: {MIN_ACADEMIC_PERCENTAGE}%)")
    print(f"\nMatch Percentage: {match_percentage}%")
    print(f"\nFinal Result: ")
    if final_decision == "Selected / Shortlisted":
        print(f"✅ {final_decision}")
    else:
        print(f"❌ {final_decision}")
    print("="*35)

# --- Choose your method below: ---

# OPTION 1: Upload a PDF file
# This will open a file uploader dialog. Select your resume PDF.
print("\n--- Please upload your resume PDF (or skip to use sample) ---")

uploaded = files.upload()

if uploaded:
    for filename, content in uploaded.items():
        print(f"Processing file: {filename}")
        run_screening_system(resume_pdf_content=content, use_sample=False)
else:
    print("No file uploaded. Proceeding with sample resume.")
    # OPTION 2: Use the predefined sample resume text
    run_screening_system(use_sample=True)

print("\nScreening process complete.")


--- Please upload your resume PDF (or skip to use sample) ---


Saving Aparajita_Mishra_Resume.pdf to Aparajita_Mishra_Resume.pdf
Saving Bichismita_Jena_Resume.pdf to Bichismita_Jena_Resume.pdf
Saving Lipsa_Sahu_Resume.pdf to Lipsa_Sahu_Resume.pdf
Saving Pragyan_Paramita_Biswal_Resume.pdf to Pragyan_Paramita_Biswal_Resume.pdf
Saving Subhasmita_Muduli_Resume.pdf to Subhasmita_Muduli_Resume.pdf
Saving Swarnamudra_Sahu_Resume.pdf to Swarnamudra_Sahu_Resume.pdf
Processing file: Aparajita_Mishra_Resume.pdf
Extracting text from uploaded PDF...

      RESUME SCREENING RESULT      
Extracted Skills: Python, Sql
Qualification: Graduate
Experience: Experience
Academic Percentage: 60% (Min required: 60%)

Match Percentage: 80%

Final Result: 
✅ Selected / Shortlisted
Processing file: Bichismita_Jena_Resume.pdf
Extracting text from uploaded PDF...

      RESUME SCREENING RESULT      
Extracted Skills: None
Qualification: N/A
Experience: N/A
Academic Percentage: 55% (Min required: 60%)

Match Percentage: 0%

Final Result: 
❌ Rejected
Processing file: Lipsa_Sahu